# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring a FAIR<sup>2</sup> dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is specified by the Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Accessing the metadata object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\nPublished: {metadata.datePublished}\nIdentifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets and fields, referencing their `@id` identifiers.

`mlcroissant` exposes record sets as part of the dataset metadata and allows inspection of their structure for precise data extraction.

We'll list all record sets and all fields/columns for each, referencing by `@id` as required.

In [ ]:
# List all RecordSet @ids in the dataset
record_sets = dataset.metadata.record_sets

print("Available RecordSets (by @id):")
for rs in record_sets:
    print(f" - {rs.id} (name: {rs.name})")
    # List associated fields, which define the columns
    print("   Fields / columns (by @id):")
    for field in rs.fields:
        print(f"     - {field.id}: {field.name} [dataType: {field.data_type}]")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Reference record set and field `@id`s as above.

In FAIR<sup>2</sup> datasets, each RecordSet may represent a core data table. Here we:
- Extract all data from all record sets detected.
- Store each as a pandas DataFrame with the RecordSet `@id` as key.

In [ ]:
# Collect data from each discovered RecordSet
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # The generator yields dicts, each record is a row
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded RecordSet {record_set_id}: Data shape {df.shape}")
    if len(df.columns) > 0:
        print("Columns:", list(df.columns))
    print()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, categorizing data, removing outliers, and grouping data by key attribute.

For demonstration, we'll perform EDA on the first record set with numeric fields, referencing each element by its `@id`.

In [ ]:
# Find the first RecordSet with numeric fields
selected_record_set_id = None
selected_numeric_field_id = None
selected_group_field_id = None
for rs in dataset.metadata.record_sets:
    for field in rs.fields:
        if getattr(field, 'data_type', None) in ['Float', 'Integer', 'Number']:
            selected_record_set_id = rs.id
            selected_numeric_field_id = field.id
            # Also try to find a group field (string/categorical)
            group_candidate = [f for f in rs.fields if getattr(f, 'data_type', None) == 'Text']
            if group_candidate:
                selected_group_field_id = group_candidate[0].id
            break
    if selected_record_set_id:
        break

# Use the IDs to operate on the correct DataFrame
print(f"Using RecordSet: {selected_record_set_id}\nNumeric field: {selected_numeric_field_id}\nGroup field: {selected_group_field_id}")

df = dataframes[selected_record_set_id]

# Try to convert the numeric column if not already numeric
df[selected_numeric_field_id] = pd.to_numeric(df[selected_numeric_field_id], errors='coerce')

# Filter records with the numeric field above a threshold
threshold = df[selected_numeric_field_id].mean() if pd.notnull(df[selected_numeric_field_id]).all() else 0
filtered_df = df[df[selected_numeric_field_id] > threshold]
print(f"Filtered records where {selected_numeric_field_id} > {threshold:.3f} (mean value):\n", filtered_df.head(3))

# Normalize numeric field
filtered_df[f"{selected_numeric_field_id}_normalized"] = (filtered_df[selected_numeric_field_id] - filtered_df[selected_numeric_field_id].mean()) / filtered_df[selected_numeric_field_id].std()
print(f"\nNormalized {selected_numeric_field_id} for filtered records:\n", filtered_df[[selected_numeric_field_id, f"{selected_numeric_field_id}_normalized"]].head(3))

# Group by a text/categorical field if available
if selected_group_field_id and selected_group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(selected_group_field_id)[selected_numeric_field_id].mean()
    print(f"\nGrouped mean by {selected_group_field_id} (first 5 results):\n", grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We will plot the distribution of the selected numeric field and, if group information is available, show means per group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(df[selected_numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {selected_numeric_field_id}")
plt.xlabel(selected_numeric_field_id)
plt.ylabel("Count")
plt.show()

# Optional: show means per group if grouping possible
if selected_group_field_id and selected_group_field_id in df.columns:
    plt.figure(figsize=(10, 4))
    group_means = df.groupby(selected_group_field_id)[selected_numeric_field_id].mean().sort_values()
    group_means.plot.bar()
    plt.title(f"Group mean {selected_numeric_field_id} by {selected_group_field_id}")
    plt.xlabel(selected_group_field_id)
    plt.ylabel(f"Mean {selected_numeric_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we've:
- Loaded FAIR<sup>2</sup>-formatted data and metadata using `mlcroissant`
- Explored record set and field structures, always referencing elements by their `@id`
- Loaded records into DataFrames for reproducibility
- Performed basic EDA including filtering, normalization, and grouping
- Visualized numeric fields and grouped summaries

Now, you can use the loaded DataFrames and the metadata structure to power further analysis or ML workflows.

**Tip:** For robust and interpretable analyses, always trace your exploration to the FAIR data schema and field `@id` definitions.